# Abdominal ultrasound, stage 1: organ recognition

Multi-label recognition of 8 abdominal organs (liver, kidney, pancreas, vessels, adrenals, gallbladder, bones, spleen)
on the Vitale et al. dataset, using the same architecture as the FER study
(VGG16 + CBAM and ViT-B/16, capsule fusion, transformer refinement). Only the output layer changes (8 sigmoid outputs).

- **Train / validation:** AUS train (simulated). **Tests:** AUS test (simulated) and the annotated RUS test scans (real).
- **Trauma context:** this is the normal-anatomy baseline for FAST (liver + kidney = RUQ / Morison's pouch,
  spleen + kidney = LUQ). The dataset has **no trauma cases**, so nothing here measures trauma detection.
- **Run order:** run the audit section first. If the palette assertion fails, update `PALETTE` in the config cell.
- Kaggle settings: GPU on, Internet on, dataset `omoikane2002/ra-us-data` attached.

In [ ]:
# transformers v5 removed its TensorFlow models (TFViTModel), so stay on v4.
# tf-keras ships with Kaggle's TensorFlow image; if the version check in the next cell fails, install the
# release that matches TensorFlow, e.g. !pip install -q "tf-keras~=2.18.0" for TF 2.18.
!pip install -q "transformers>=4.40,<5"

In [ ]:
# Run this cell before anything imports TensorFlow (restart the kernel if TF was already imported).
# The HF TFViTModel is a tf-keras (Keras 2) model. Inside a Keras 3 model its weights are not tracked,
# so the ViT silently never trains. Running the whole notebook on tf-keras makes it a proper sub-layer.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import random
import numpy as np
import tensorflow as tf
import transformers

import tf_keras

# tf.keras has no __version__ under tf-keras, so compare the classes instead
assert tf.keras.Model is tf_keras.Model, (
    "tf.keras is not tf-keras (Keras 2); restart the kernel and run this cell before any TensorFlow import"
)
assert int(transformers.__version__.split(".")[0]) < 5, "transformers>=5 has no TensorFlow models"
print("TensorFlow:", tf.__version__, "| tf-keras:", tf_keras.__version__, "| transformers:", transformers.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Configuration

In [ ]:
import os
import re
import glob
import shutil
import hashlib
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =================================
# Paths
# =================================
DATA_ROOT = '/kaggle/input/datasets/omoikane2002/ra-us-data/abdominal_US'
if not os.path.isdir(DATA_ROOT):
    # Kaggle mounts datasets at different depths depending on the image version
    found = sorted(glob.glob('/kaggle/input/**/abdominal_US', recursive=True))
    assert found, "Could not find abdominal_US under /kaggle/input; attach the ra-us-data dataset"
    DATA_ROOT = found[0]
print("DATA_ROOT:", DATA_ROOT)

# name: (image folder, mask folder or None)
SETS = {
    "aus_train": ("AUS/images/train", "AUS/annotations/train"),
    "aus_test": ("AUS/images/test", "AUS/annotations/test"),
    "rus_test": ("RUS/images/test", "RUS/annotations/test"),
    "rus_train": ("RUS/images/train", None),  # no masks: qualitative gallery only
}
WORK_DIR = '/kaggle/working/us'
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

# =================================
# Labels
# =================================
ORGANS = ["liver", "kidney", "pancreas", "vessels", "adrenals", "gallbladder", "bones", "spleen"]

# Nominal RGB values for the dataset's color legend. The audit section checks them against the real masks;
# if it fails, replace them with the printed colors.
PALETTE = {
    "liver": (255, 0, 255),        # violet
    "kidney": (255, 255, 0),       # yellow
    "pancreas": (0, 0, 255),       # blue
    "vessels": (255, 0, 0),        # red
    "adrenals": (0, 255, 255),     # light blue
    "gallbladder": (0, 255, 0),    # green
    "bones": (255, 255, 255),      # white
    "spleen": (255, 192, 203),     # pink
}
BACKGROUND = (0, 0, 0)
COLOR_TOL = 60                  # max RGB distance for a pixel to count as a legend color (tolerates JPEG noise)
MIN_ORGAN_FRAC = 0.005          # an organ is "present" if it covers at least 0.5% of the mask
SIGNIFICANT_COLOR_FRAC = 0.001  # colors covering >= 0.1% of all mask pixels must match the legend

# =================================
# Training
# =================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
VAL_FRAC = 0.15
GROUP_REGEX = None  # e.g. r'^(\d+)_' once the audit shows how file names encode the CT volume
MAX_REPEAT = 5      # cap for rare-organ repeat sampling

## 2. Data audit
Run this section first. It checks file pairing and that every mask color matches `PALETTE`.

In [ ]:
# =================================
# Data audit: file pairing, image sizes, mask colors
# =================================
MASK_SUFFIXES = ("_mask", "_label", "_labels", "_annotation", "_annot", "_seg", "_gt")

def stem_key(filename):
    """File stem used to pair an image with its mask (common mask suffixes removed)."""
    stem = os.path.splitext(filename)[0].lower()
    for suffix in MASK_SUFFIXES:
        if stem.endswith(suffix):
            return stem[: -len(suffix)]
    return stem

def list_images(directory):
    return sorted(f for f in os.listdir(directory) if f.lower().endswith(IMG_EXTS))

def pair_images_and_masks(image_dir, mask_dir):
    """Returns (image/mask path pairs, image keys without mask, mask keys without image)."""
    images = {stem_key(f): f for f in list_images(image_dir)}
    masks = {stem_key(f): f for f in list_images(mask_dir)}
    common = sorted(images.keys() & masks.keys())
    pairs = [(os.path.join(image_dir, images[k]), os.path.join(mask_dir, masks[k])) for k in common]
    return pairs, sorted(images.keys() - masks.keys()), sorted(masks.keys() - images.keys())

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    assert img is not None, f"Could not read {path}"
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def load_mask(path, max_side=256):
    """RGB mask, downscaled with nearest-neighbour so no new colors are created."""
    mask = read_rgb(path)
    scale = max_side / max(mask.shape[:2])
    if scale < 1:
        mask = cv2.resize(mask, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST)
    return mask

def legend():
    """Legend names and RGB values from the current PALETTE (background first)."""
    names = ["background"] + ORGANS
    rgb = np.array([BACKGROUND] + [PALETTE[o] for o in ORGANS], dtype=np.float32)
    return names, rgb

def nearest_legend(colors):
    """For each RGB color (N, 3): index of the nearest legend entry and its distance."""
    _, rgb = legend()
    d = np.linalg.norm(np.asarray(colors, dtype=np.float32)[:, None, :] - rgb[None, :, :], axis=-1)
    return d.argmin(axis=1), d.min(axis=1)

def mask_color_counts(mask):
    colors, counts = np.unique(mask.reshape(-1, 3), axis=0, return_counts=True)
    return colors, counts

def audit_mask_colors(mask_paths, title, top=25, bin_size=32):
    """
    Print the dominant mask colors and return the color groups that match no legend entry.
    Near-identical colors are grouped (bins of bin_size per channel), so JPEG noise doesn't split one legend
    color into thousands of rows, each too small to notice.
    """
    totals = Counter()
    for p in mask_paths:
        colors, counts = mask_color_counts(load_mask(p))
        for c, n in zip(map(tuple, colors), counts):
            totals[c] += int(n)
    colors = np.array(list(totals.keys()), dtype=np.float32)
    shares = np.array(list(totals.values()), dtype=np.float64) / sum(totals.values())
    idx, dist = nearest_legend(colors)
    names, _ = legend()

    groups = {}
    for i, key in enumerate(map(tuple, (colors // bin_size).astype(int))):
        groups.setdefault(key, []).append(i)
    rows = []
    for members in map(np.array, groups.values()):
        rep = members[np.argmax(shares[members])]  # most frequent exact color in the group
        rows.append({
            "rgb": tuple(int(v) for v in colors[rep]),
            "share_%": 100 * shares[members].sum(),
            "unmatched_%": 100 * shares[members][dist[members] > COLOR_TOL].sum(),
            "nearest": names[idx[rep]],
            "distance": round(float(dist[rep]), 1),
        })
    table = pd.DataFrame(rows).sort_values("share_%", ascending=False).reset_index(drop=True)
    print(f"\n{title}: {len(totals)} distinct colors in {len(table)} groups, top {top} groups by pixel share")
    print(table.head(top).round(3).to_string(index=False))
    print(f"Pixels farther than COLOR_TOL from every legend color: {100 * shares[dist > COLOR_TOL].sum():.2f}%")
    return table[table["unmatched_%"] >= 100 * SIGNIFICANT_COLOR_FRAC]

paired = {}
audit_failures = []
for name, (img_rel, mask_rel) in SETS.items():
    img_dir = os.path.join(DATA_ROOT, img_rel)
    n_images = len(list_images(img_dir))
    if mask_rel is None:
        print(f"\n{name}: {n_images} images, no masks")
        continue
    pairs, img_only, mask_only = pair_images_and_masks(img_dir, os.path.join(DATA_ROOT, mask_rel))
    paired[name] = pairs
    print(f"\n{name}: {n_images} images, {len(pairs)} with a mask, "
          f"{len(img_only)} images without mask, {len(mask_only)} masks without image")
    print("   sample image names:", [os.path.basename(p[0]) for p in pairs[:5]])
    print("   sample mask names: ", [os.path.basename(p[1]) for p in pairs[:5]])
    if mask_only:
        print("   masks without image (first 5):", mask_only[:5])
    unmatched = audit_mask_colors([m for _, m in pairs], name)
    if len(unmatched):
        audit_failures.append((name, unmatched))

# Image and mask sizes (first 30 pairs per set)
for name, pairs in paired.items():
    img_shapes = Counter(cv2.imread(i, cv2.IMREAD_UNCHANGED).shape for i, _ in pairs[:30])
    mask_shapes = Counter(cv2.imread(m, cv2.IMREAD_UNCHANGED).shape for _, m in pairs[:30])
    print(f"\n{name} image shapes: {dict(img_shapes)} | mask shapes: {dict(mask_shapes)}")

assert all(len(p) > 0 for p in paired.values()), "A set has no image/mask pairs; check the file names printed above"
if audit_failures:
    for name, unmatched in audit_failures:
        print(f"\n{name}: significant mask colors that match no PALETTE entry:")
        print(unmatched[["rgb", "unmatched_%"]].round(3).to_string(index=False))
    raise AssertionError("Mask colors don't match PALETTE. Update PALETTE in the config cell "
                         "(or paste this output back) and rerun from the config cell.")
print("\nAudit passed: every significant mask color matches a PALETTE entry.")

In [ ]:
# Visual check: the legend, then image / mask / overlay examples from each annotated set
names, rgb = legend()
plt.figure(figsize=(12, 1.5))
for i, (n, c) in enumerate(zip(names, rgb)):
    plt.fill_between([i, i + 0.9], 0, 1, color=c / 255.0, edgecolor="grey")
    plt.text(i + 0.45, -0.25, n, ha="center", fontsize=9)
plt.xlim(0, len(names))
plt.axis("off")
plt.title("PALETTE as configured (compare with the mask colors below)")
plt.show()

rng = np.random.default_rng(SEED)
for name, pairs in paired.items():
    picks = rng.choice(len(pairs), size=min(3, len(pairs)), replace=False)
    fig, axes = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)))
    axes = np.atleast_2d(axes)
    for row, k in enumerate(picks):
        img_path, mask_path = pairs[k]
        image = read_rgb(img_path)
        mask = cv2.resize(read_rgb(mask_path), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        overlay = (0.6 * image + 0.4 * mask).astype(np.uint8)
        for ax, im, title in zip(axes[row], (image, mask, overlay), ("image", "mask", "overlay")):
            ax.imshow(im)
            ax.set_title(f"{name}: {title}\n{os.path.basename(img_path)}", fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

## 3. Labels, preprocessing and splits

In [ ]:
# =================================
# Organ labels from the masks
# =================================

def mask_to_organ_fractions(mask):
    """Fraction of the mask covered by each organ (pixels matched to the nearest legend color within COLOR_TOL)."""
    colors, counts = mask_color_counts(mask)
    idx, dist = nearest_legend(colors)
    names, _ = legend()
    total = counts.sum()
    return {o: counts[(idx == names.index(o)) & (dist <= COLOR_TOL)].sum() / total for o in ORGANS}

def extract_group(path):
    """CT-volume / subject ID from the file name, if GROUP_REGEX is set and matches."""
    if GROUP_REGEX is None:
        return None
    match = re.search(GROUP_REGEX, os.path.basename(path))
    return match.group(1) if match else None

def build_label_frame(pairs, set_name):
    rows = []
    for img_path, mask_path in pairs:
        fractions = mask_to_organ_fractions(load_mask(mask_path))
        row = {"set": set_name, "src_path": img_path, "mask_path": mask_path, "group": extract_group(img_path)}
        for o in ORGANS:
            row[f"{o}_frac"] = fractions[o]
            row[o] = int(fractions[o] >= MIN_ORGAN_FRAC)
        rows.append(row)
    return pd.DataFrame(rows)

label_frames = {name: build_label_frame(pairs, name) for name, pairs in paired.items()}

print("Organ prevalence (share of images containing the organ):")
print(pd.DataFrame({name: df[ORGANS].mean() for name, df in label_frames.items()}).round(3))
print("\nOrgan counts:")
print(pd.DataFrame({name: df[ORGANS].sum() for name, df in label_frames.items()}))

missing_in_train = [o for o in ORGANS if label_frames["aus_train"][o].sum() == 0]
if missing_in_train:
    print(f"\nWARNING: never present in AUS train, so the model cannot learn them: {missing_in_train}")

# The RUS masks only cover some organs: an organ missing from a partial annotation is not missing from the image.
# Score each test set only on organs that have both positives and negatives there (AUC needs both).
RUS_ANNOTATED_ORGANS = [o for o in ORGANS if label_frames["rus_test"][o].sum() > 0]
EVAL_ORGANS = {
    "aus_test": [o for o in ORGANS if label_frames["aus_test"][o].nunique() == 2],
    "rus_test": [o for o in RUS_ANNOTATED_ORGANS if label_frames["rus_test"][o].nunique() == 2],
}
for name, organs in EVAL_ORGANS.items():
    print(f"\nOrgans scored on {name}: {organs}")
    print(f"   not scored: {[o for o in ORGANS if o not in organs]}")

In [ ]:
# =================================
# CLAHE preprocessing (same function as the FER study)
# =================================

def preprocess_image(image_path, target_size=(224, 224)):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, target_size)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

def preprocess_frame(df, set_name):
    """Write CLAHE-processed copies to WORK_DIR/<set_name>/ and add their paths as the 'path' column."""
    out_dir = os.path.join(WORK_DIR, set_name)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir)
    paths = []
    for src in df["src_path"]:
        out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(src))[0] + ".png")
        cv2.imwrite(out_path, preprocess_image(src, IMG_SIZE))
        paths.append(out_path)
    assert len(set(paths)) == len(paths), f"{set_name}: two source files map to the same output name"
    df = df.copy()
    df["path"] = paths
    return df

for name in list(label_frames):
    label_frames[name] = preprocess_frame(label_frames[name], name)

# Unannotated real scans, for the qualitative gallery only
rus_train_dir = os.path.join(DATA_ROOT, SETS["rus_train"][0])
rus_unlabeled_df = preprocess_frame(
    pd.DataFrame({"src_path": [os.path.join(rus_train_dir, f) for f in list_images(rus_train_dir)]}), "rus_train")
print({name: len(df) for name, df in label_frames.items()}, "| unannotated RUS:", len(rus_unlabeled_df))

In [ ]:
# =================================
# Train / validation split of AUS train, plus duplicate checks
# =================================
from sklearn.model_selection import GroupShuffleSplit, train_test_split

aus_train_all = label_frames["aus_train"]
if aus_train_all["group"].notna().all():
    splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
    train_idx, val_idx = next(splitter.split(aus_train_all, groups=aus_train_all["group"]))
    print(f"Volume-wise split over {aus_train_all['group'].nunique()} groups")
else:
    print("WARNING: no CT-volume IDs (GROUP_REGEX unset or not matching), so this is a random split. "
          "Neighbouring slices of one volume can land in both train and val, which makes val scores optimistic. "
          "AUS test and RUS test are not affected.")
    train_idx, val_idx = train_test_split(np.arange(len(aus_train_all)), test_size=VAL_FRAC, random_state=SEED)

train_df = aus_train_all.iloc[train_idx].reset_index(drop=True)
val_df = aus_train_all.iloc[val_idx].reset_index(drop=True)
aus_test_df = label_frames["aus_test"]
rus_test_df = label_frames["rus_test"]
if aus_train_all["group"].notna().all():
    assert not set(train_df["group"]) & set(val_df["group"]), "A CT volume appears in both train and val"
print(f"train {len(train_df)} | val {len(val_df)} | AUS test {len(aus_test_df)} | RUS test {len(rus_test_df)}")
print("\nPrevalence train vs val:")
print(pd.DataFrame({"train": train_df[ORGANS].mean(), "val": val_df[ORGANS].mean()}).round(3))

def md5_set(paths):
    hashes = set()
    for p in paths:
        with open(p, "rb") as fh:
            hashes.add(hashlib.md5(fh.read()).hexdigest())
    return hashes

def report_overlap(paths_a, paths_b, name_a, name_b):
    """Informational: number of byte-identical source images shared by two sets."""
    overlap = len(md5_set(paths_a) & md5_set(paths_b))
    print(f"Byte-identical images shared by {name_a} and {name_b}: {overlap}")
    return overlap

report_overlap(train_df["src_path"], val_df["src_path"], "train", "val")
report_overlap(aus_train_all["src_path"], aus_test_df["src_path"], "AUS train", "AUS test")
report_overlap(aus_train_all["src_path"], rus_test_df["src_path"], "AUS train", "RUS test")

In [ ]:
# =================================
# Rare-organ repeat sampling (the multi-label version of class oversampling)
# =================================

def repeat_factors(df, organs, max_repeat=MAX_REPEAT):
    """
    How many times each training row is repeated: the largest (max_count // organ_count) over the organs in
    that image, capped at max_repeat. Rows are repeated in the dataframe only; augmentation happens on the fly,
    so no files are written.
    """
    counts = df[organs].sum()
    max_count = counts.max()
    per_organ = {o: (max_count // counts[o]) if counts[o] > 0 else 1 for o in organs}
    factors = []
    for _, row in df.iterrows():
        present = [per_organ[o] for o in organs if row[o] == 1]
        factors.append(int(np.clip(max(present) if present else 1, 1, max_repeat)))
    return np.array(factors)

factors = repeat_factors(train_df, ORGANS)
train_df_rep = train_df.loc[train_df.index.repeat(factors)].reset_index(drop=True)
print(f"Training rows: {len(train_df)} -> {len(train_df_rep)} after repeat sampling")
print(pd.DataFrame({"before": train_df[ORGANS].sum(), "after": train_df_rep[ORGANS].sum()}))

# Focal-loss weight of the positive term per organ: rare organs get more weight
alpha_vec = np.clip(1.0 - train_df_rep[ORGANS].mean().values, 0.25, 0.9).astype("float32")
print("\nFocal alpha per organ:", dict(zip(ORGANS, alpha_vec.round(3))))

## 4. Model (same architecture as the FER study)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout,
    BatchNormalization, Conv2D, Multiply,
    Lambda, Concatenate, Reshape, Input
)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    f1_score, precision_recall_curve, hamming_loss
)

from transformers import TFViTModel

In [ ]:

class ViTWrapper(tf.keras.layers.Layer):
    """
    A custom Keras layer that wraps TFViTModel and ensures compatibility with the functional API.
    """
    def __init__(self, pretrained_name="google/vit-base-patch16-224-in21k", **kwargs):
        super().__init__(**kwargs)
        self.pretrained_name = pretrained_name
        self.vit_model = TFViTModel.from_pretrained(pretrained_name)

    def call(self, inputs, training=False):
        # Transpose to channels-first for ViT
        x = tf.transpose(inputs, perm=[0, 3, 1, 2])  # (B, 3, 224, 224)
        # Normalize inputs: the in21k checkpoint was trained with mean = std = 0.5
        x = (x - 0.5) / 0.5
        outputs = self.vit_model(pixel_values=x, training=training)
        return outputs.pooler_output

    def freeze(self):
        """Freeze the whole ViT. The model-level flag overrides every sub-layer."""
        self.vit_model.trainable = False

    def unfreeze_layers(self):
        """Unfreeze the whole ViT."""
        self.vit_model.trainable = True

    def get_config(self):
        config = super().get_config()
        config.update({"pretrained_name": self.pretrained_name})
        return config


In [ ]:
# -----------------------------
# Squash Activation Function for Capsules
# -----------------------------
def squash(s, axis=-1, epsilon=1e-7):
    s_squared_norm = tf.reduce_sum(tf.square(s), axis=axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm)
    return scale * s / tf.sqrt(s_squared_norm + epsilon)

# -----------------------------
# Advanced Capsule Fusion Block (CFB)
# -----------------------------
class AdvancedCapsuleFusionBlock(tf.keras.layers.Layer):
    """
    Advanced Capsule Fusion Block that fuses two feature vectors using an iterative dynamic routing
    algorithm with enhanced transformations, residual updates, dropout, and batch normalization.
    """
    def __init__(self, capsule_dim=256, num_iterations=2, dropout_rate=0.4, **kwargs):
        super(AdvancedCapsuleFusionBlock, self).__init__(**kwargs)
        self.capsule_dim = capsule_dim
        self.num_iterations = num_iterations
        self.dropout_rate = dropout_rate
        
        # Two-layer MLP for each branch's transformation
        self.mlp1 = tf.keras.Sequential([
            Dense(capsule_dim, activation="relu"),
            Dropout(dropout_rate),
            Dense(capsule_dim, activation=None)
        ])
        self.bn1 = BatchNormalization()
        
        self.mlp2 = tf.keras.Sequential([
            Dense(capsule_dim, activation="relu"),
            Dropout(dropout_rate),
            Dense(capsule_dim, activation=None)
        ])
        self.bn2 = BatchNormalization()
    
    def call(self, inputs, training=False):
        # inputs: list [f1, f2] from two branches
        f1, f2 = inputs
        # Transform each branch using a two-layer MLP
        u1 = self.bn1(self.mlp1(f1), training=training)  # (batch, capsule_dim)
        u2 = self.bn2(self.mlp2(f2), training=training)  # (batch, capsule_dim)
        
        # Stack to form capsules: shape (batch, 2, capsule_dim)
        u = tf.stack([u1, u2], axis=1)
        
        # Initialize routing logits b (batch, 2) with zeros
        b = tf.zeros(shape=(tf.shape(u)[0], 2), dtype=tf.float32)
        
        # Iterative dynamic routing
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=1)  # coupling coefficients (batch, 2)
            c_expanded = tf.expand_dims(c, axis=-1)  # (batch, 2, 1)
            # Weighted sum of capsule predictions
            s = tf.reduce_sum(c_expanded * u, axis=1)  # (batch, capsule_dim)
            v = squash(s, axis=-1)  # (batch, capsule_dim)
            
            # Residual update: encourage consistency with previous capsule output
            if i > 0:
                v = 0.5 * v + 0.5 * v_prev
            v_prev = v  # store for residual update
            
            v_expanded = tf.expand_dims(v, axis=1)  # (batch, 1, capsule_dim)
            # Update routing logits based on agreement
            b += tf.reduce_sum(u * v_expanded, axis=-1)  # (batch, 2)
        return v  # Final fused capsule (batch, capsule_dim)


In [ ]:
# -----------------------------
# Positional Embedding Layer for TRB
# -----------------------------
class PositionalEmbedding(tf.keras.layers.Layer):
    """
    Learns a positional embedding for a fixed number of tokens.
    """
    def __init__(self, num_tokens, token_dim, **kwargs):
        super(PositionalEmbedding, self).__init__(**kwargs)
        self.num_tokens = num_tokens
        self.token_dim = token_dim

    def build(self, input_shape):
        # Create a learnable positional embedding matrix
        self.pos_embedding = self.add_weight(
            shape=(1, self.num_tokens, self.token_dim),
            initializer="uniform",
            trainable=True,
            name="pos_embedding"
        )
        super(PositionalEmbedding, self).build(input_shape)

    def call(self, inputs):
        # inputs shape: (batch, num_tokens, token_dim)
        return inputs + self.pos_embedding

# -----------------------------
# Advanced Transformer Encoder Block for TRB
# -----------------------------
class TransformerEncoderBlock(tf.keras.layers.Layer):
    """
    A full transformer encoder block with multi-head self-attention, feed-forward network,
    residual connections, and layer normalization.
    """
    def __init__(self, token_dim, num_heads, ff_dim, dropout_rate=0.4, **kwargs):
        super(TransformerEncoderBlock, self).__init__(**kwargs)
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=token_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(token_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
    
    def call(self, inputs, training=False):
        attn_output = self.mha(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# -----------------------------
# Advanced Transformer Refinement Block (TRB)
# -----------------------------
class AdvancedTransformerRefinementBlock(tf.keras.layers.Layer):
    """
    Refines the fused capsule features using a stack of transformer encoder blocks.
    The fused vector is projected into multiple tokens, positional embeddings are added,
    processed by transformer layers, and then aggregated.
    """
    def __init__(self, num_tokens=4, token_dim=64, num_heads=4, ff_dim=128, num_layers=2, dropout_rate=0.3, **kwargs):
        super(AdvancedTransformerRefinementBlock, self).__init__(**kwargs)
        self.num_tokens = num_tokens
        self.token_dim = token_dim
        self.proj = Dense(num_tokens * token_dim)
        self.reshape_layer = Lambda(lambda x: tf.reshape(x, (-1, num_tokens, token_dim)))
        self.pos_emb = PositionalEmbedding(num_tokens, token_dim)
        self.transformer_layers = [TransformerEncoderBlock(token_dim, num_heads, ff_dim, dropout_rate) for _ in range(num_layers)]
    
    def call(self, inputs, training=False):
        # inputs: fused capsule features (batch, capsule_dim)
        tokens = self.proj(inputs)                # (batch, num_tokens * token_dim)
        tokens = self.reshape_layer(tokens)         # (batch, num_tokens, token_dim)
        tokens = self.pos_emb(tokens)               # Add positional embedding
        for layer in self.transformer_layers:
            tokens = layer(tokens, training=training)
        # Aggregate tokens via mean pooling
        refined = tf.reduce_mean(tokens, axis=1)    # (batch, token_dim)
        return refined

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class MixupSequence(Sequence):
    def __init__(self, datagen, alpha=0.1, **kwargs):
        """
        Wraps a Keras ImageDataGenerator (Sequence) to perform MixUp on each batch.

        Args:
            datagen: A Keras ImageDataGenerator Sequence (e.g., from flow_from_directory).
            alpha: Beta distribution parameter for mixing. Higher alpha means stronger mixing.
        """
        super().__init__(**kwargs)
        self.datagen = datagen
        self.alpha = alpha
        self.batch_size = datagen.batch_size
        self.steps = len(datagen)

    def __len__(self):
        # Return the number of batches per epoch
        return self.steps

    def __getitem__(self, idx):
        # Get the idx-th batch from the underlying generator
        batch_x, batch_y = self.datagen[idx]
        batch_size = batch_x.shape[0]

        # Sample lambda from Beta(alpha, alpha)
        lam = np.random.beta(self.alpha, self.alpha)

        # Shuffle batch indices
        index = np.random.permutation(batch_size)

        # Perform MixUp on images and labels
        mixed_x = lam * batch_x + (1 - lam) * batch_x[index, :]
        mixed_y = lam * batch_y + (1 - lam) * batch_y[index, :]

        return mixed_x, mixed_y

    def on_epoch_end(self):
        # Let the wrapped generator reshuffle; otherwise every epoch repeats the first epoch's order
        self.datagen.on_epoch_end()


In [ ]:
def binary_focal_loss(alpha, gamma=2.0):
    """
    Focal loss for multi-label classification (one sigmoid per organ).

    Args:
        alpha: per-organ weight of the positive term; negatives get 1 - alpha.
        gamma: focusing parameter that down-weights easy examples.

    Returns:
        A loss function that can be used in model.compile().
    """
    alpha = tf.constant(alpha, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-4, 1 - 1e-4)
        positive = -alpha * y_true * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
        negative = -(1 - alpha) * (1 - y_true) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
        return tf.reduce_mean(tf.reduce_sum(positive + negative, axis=-1))

    return loss_fn

In [ ]:
# =================================
# CBAM + VGG16 + ViT (Multi-Branch) Model, same as the FER study with a sigmoid output
# =================================

def attention_block(input_tensor):
    # Channel Attention
    avg_pool = GlobalAveragePooling2D()(input_tensor)  # (batch, channels)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=[1, 2], keepdims=False))(input_tensor)  # (batch, channels)

    reduced_channels = input_tensor.shape[-1] // 8  # Reduce dimensionality
    avg_dense = Dense(reduced_channels, activation="relu")(avg_pool)
    max_dense = Dense(reduced_channels, activation="relu")(max_pool)
    channel_attention = Dense(input_tensor.shape[-1], activation="sigmoid")(avg_dense + max_dense)
    channel_refined = Multiply()([input_tensor, Reshape((1, 1, -1))(channel_attention)])

    # Spatial Attention
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(channel_refined)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(channel_refined)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    spatial_attention = Conv2D(1, (7, 7), activation="sigmoid", padding="same")(concat)
    refined_output = Multiply()([channel_refined, spatial_attention])
    return refined_output


class VGGPreprocess(tf.keras.layers.Layer):
    """
    Converts the [0, 1] RGB model input into the caffe preprocessing VGG16's ImageNet weights
    were trained with: 0-255 range, BGR channel order, ImageNet mean subtracted.
    """
    def call(self, inputs):
        x = inputs * 255.0
        x = x[..., ::-1]  # RGB -> BGR
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)


def build_vgg16_vit_model(num_outputs, output_activation="sigmoid"):
    """
    Creates a multi-branch model that fuses features from:
      - VGG16 + CBAM CNN branch
      - Vision Transformer (ViT) branch
    Fusion is performed using an advanced Capsule Fusion Block followed by an advanced Transformer Refinement Block.
    """
    input_tensor = Input(shape=(224, 224, 3), name="image_input")

    # VGG16 + CBAM Branch
    base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers[:-8]:
        layer.trainable = False
    vgg_input = VGGPreprocess(name="vgg_preprocess")(input_tensor)
    vgg_features = base_model(vgg_input)
    vgg_features = attention_block(vgg_features)
    vgg_features = GlobalAveragePooling2D()(vgg_features)
    vgg_features = Dense(256, activation="relu")(vgg_features)

    # ViT Branch (normalizes its own input)
    vit_branch = ViTWrapper(pretrained_name="google/vit-base-patch16-224-in21k", name="vit_wrapper")
    vit_features = vit_branch(input_tensor)

    # Fuse features using the Advanced Capsule Fusion Block
    fused_capsule = AdvancedCapsuleFusionBlock(capsule_dim=256, num_iterations=2, dropout_rate=0.4)([vgg_features, vit_features])

    # Refine fused features with the Advanced Transformer Refinement Block
    refined_features = AdvancedTransformerRefinementBlock(num_tokens=4, token_dim=64, num_heads=4, ff_dim=128, num_layers=2, dropout_rate=0.4)(fused_capsule)

    # Output layer: one sigmoid per organ
    final_output = Dense(num_outputs, activation=output_activation)(refined_features)

    model = Model(inputs=input_tensor, outputs=final_output, name="Advanced_VGG16_CBAM_ViT_Model")
    return model


def count_vit_trainable_weights(model):
    """Number of ViT weight tensors the optimizer will update (0 means the ViT is frozen)."""
    n_vit = len(model.get_layer("vit_wrapper").trainable_weights)
    print(f"Trainable weight tensors: ViT {n_vit}, whole model {len(model.trainable_weights)}")
    return n_vit

# Parallel batch loading: ImageDataGenerator augmentation runs on the CPU
FIT_KWARGS = dict(workers=4, max_queue_size=20)
CHECKPOINT_DIR = "/kaggle/working/checkpoints_us"

def make_callbacks(phase):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    return [
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        # Best weights on disk, in case the Kaggle session times out mid-run
        ModelCheckpoint(
            os.path.join(CHECKPOINT_DIR, f"phase{phase}_best"),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True
        ),
    ]

## 5. Training

In [ ]:
# =================================
# Data generators and two-phase training (same schedule as the FER study)
# =================================

# Train augmentation: FER settings without horizontal flips (they swap anatomical sides).
# The area outside the ultrasound fan is black, so fill with black instead of mirroring anatomy into it.
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.0],
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=False,
    fill_mode='constant',
    cval=0.0
)
# Validation / test: rescale only
plain_datagen = ImageDataGenerator(rescale=1.0/255.0)

def make_generator(datagen, df, shuffle, seed=None):
    generator = datagen.flow_from_dataframe(
        df,
        x_col="path",
        y_col=ORGANS,
        class_mode="raw",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=seed
    )
    assert generator.n == len(df), "Some image paths were rejected by flow_from_dataframe"
    return generator

train_generator = make_generator(train_datagen, train_df_rep, shuffle=True, seed=SEED)
val_generator = make_generator(plain_datagen, val_df, shuffle=False)

loss_fn = binary_focal_loss(alpha_vec, gamma=2.0)

def compile_model(model, learning_rate):
    model.compile(
        optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-3),
        loss=loss_fn,
        metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc"),
                 tf.keras.metrics.BinaryAccuracy(name="binary_accuracy")]
    )

def train_model():
    model = build_vgg16_vit_model(len(ORGANS), output_activation="sigmoid")
    mixup_train_sequence = MixupSequence(train_generator, alpha=0.1)

    # Phase 1: ViT frozen; VGG16 blocks 4-5, CBAM, fusion blocks and head train
    model.get_layer("vit_wrapper").freeze()
    assert count_vit_trainable_weights(model) == 0
    compile_model(model, 3e-5)
    print("\n=== Phase 1 Training ===")
    history_phase1 = model.fit(
        mixup_train_sequence,
        validation_data=val_generator,
        epochs=25,
        callbacks=make_callbacks(phase=1),
        **FIT_KWARGS
    )

    # Phase 2: unfreeze everything, including VGG16 blocks 1-3 and the ViT
    for layer in model.layers:
        layer.trainable = True
    model.get_layer("vit_wrapper").unfreeze_layers()
    assert count_vit_trainable_weights(model) > 0, "ViT weights are not trainable; check the tf.keras version cell"
    compile_model(model, 1e-5)
    print("\n=== Phase 2 Fine-tuning ===")
    history_phase2 = model.fit(
        mixup_train_sequence,
        validation_data=val_generator,
        epochs=35,
        callbacks=make_callbacks(phase=2),
        **FIT_KWARGS
    )

    history = {k: history_phase1.history[k] + history_phase2.history[k] for k in ("loss", "val_loss", "auc", "val_auc")}
    plt.figure(figsize=(12, 5))
    for i, metric in enumerate(("auc", "loss")):
        plt.subplot(1, 2, i + 1)
        plt.plot(history[metric], label=f"Train {metric}")
        plt.plot(history[f"val_{metric}"], label=f"Validation {metric}")
        plt.axvline(len(history_phase1.history["loss"]) - 0.5, color="grey", linestyle=":", label="Phase 2 starts")
        plt.title(f"Model {metric}")
        plt.xlabel("Epochs")
        plt.legend()
    plt.show()
    return model

final_model = train_model()

## 6. Evaluation

In [ ]:
# =================================
# Evaluation: per-organ metrics on AUS test (simulated) and RUS test (real)
# =================================

def predict_frame(model, df, datagen=plain_datagen):
    """Sigmoid outputs for every row of df, in order."""
    return model.predict(make_generator(datagen, df, shuffle=False), verbose=1)

def tune_thresholds(y_true, y_prob, organs):
    """Per-organ threshold that maximizes F1 on validation (0.5 if the organ is all-positive or all-negative)."""
    thresholds = {}
    for j, organ in enumerate(organs):
        n_pos = y_true[:, j].sum()
        if n_pos == 0 or n_pos == len(y_true):
            thresholds[organ] = 0.5
            continue
        precision, recall, cutoffs = precision_recall_curve(y_true[:, j], y_prob[:, j])
        f1 = 2 * precision * recall / np.clip(precision + recall, 1e-8, None)
        thresholds[organ] = float(cutoffs[np.argmax(f1[:-1])])
    return thresholds

def bootstrap_macro_auc(y_true, y_prob, n_boot=1000, seed=SEED):
    """95% CI of the macro AUC by resampling images (organs without both classes in a resample are skipped)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        valid = [j for j in range(yt.shape[1]) if 0 < yt[:, j].sum() < n]
        if valid:
            scores.append(np.mean([roc_auc_score(yt[:, j], yp[:, j]) for j in valid]))
    return np.percentile(scores, [2.5, 97.5])

def evaluate_multilabel(y_true, y_prob, organs, thresholds, name):
    """Per-organ table, macro AUC with bootstrap CI, exact match, Hamming loss and ROC curves."""
    y_pred = np.stack([(y_prob[:, j] >= thresholds[o]).astype(int) for j, o in enumerate(organs)], axis=1)
    rows = []
    for j, organ in enumerate(organs):
        t, p, s = y_true[:, j], y_pred[:, j], y_prob[:, j]
        tp, fn = int(((t == 1) & (p == 1)).sum()), int(((t == 1) & (p == 0)).sum())
        tn, fp = int(((t == 0) & (p == 0)).sum()), int(((t == 0) & (p == 1)).sum())
        rows.append({
            "organ": organ,
            "positives": int(t.sum()),
            "negatives": int(len(t) - t.sum()),
            "AUC": roc_auc_score(t, s),
            "AP": average_precision_score(t, s),
            "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1),
            "F1": f1_score(t, p, zero_division=0),
            "threshold": thresholds[organ],
        })
    table = pd.DataFrame(rows).set_index("organ").round(3)
    macro_auc = float(table["AUC"].mean())
    ci_low, ci_high = bootstrap_macro_auc(y_true, y_prob)
    print(f"\n===== {name}: {len(y_true)} images; organs scored: {', '.join(organs)} =====")
    print(table.to_string())
    print(f"Macro AUC {macro_auc:.3f} (95% CI {ci_low:.3f}-{ci_high:.3f}) | "
          f"exact match {(y_pred == y_true).all(axis=1).mean():.3f} | Hamming loss {hamming_loss(y_true, y_pred):.3f}")

    plt.figure(figsize=(8, 6))
    for j, organ in enumerate(organs):
        fpr, tpr, _ = roc_curve(y_true[:, j], y_prob[:, j])
        plt.plot(fpr, tpr, label=f"{organ} (AUC = {table.loc[organ, 'AUC']:.2f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random chance")
    plt.title(f"ROC curves: {name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.grid()
    plt.show()
    return table, macro_auc, (ci_low, ci_high)

def score_set(probs, df, organs, name):
    """Evaluate the model's probabilities for df on the given subset of organs."""
    cols = [ORGANS.index(o) for o in organs]
    return evaluate_multilabel(df[organs].values, probs[:, cols], organs, thresholds, name)

# Thresholds are tuned on validation, then applied unchanged to both test sets
val_probs = predict_frame(final_model, val_df)
thresholds = tune_thresholds(val_df[ORGANS].values, val_probs, ORGANS)
print("Per-organ thresholds (tuned on validation):", {o: round(t, 3) for o, t in thresholds.items()})

aus_probs = predict_frame(final_model, aus_test_df)
rus_probs = predict_frame(final_model, rus_test_df)
aus_table, aus_macro, aus_ci = score_set(aus_probs, aus_test_df, EVAL_ORGANS["aus_test"], "AUS test (simulated)")
rus_table, rus_macro, rus_ci = score_set(rus_probs, rus_test_df, EVAL_ORGANS["rus_test"], "RUS test (real)")

# Sim-to-real gap per organ
comparison = pd.DataFrame({"AUS test (simulated)": aus_table["AUC"], "RUS test (real)": rus_table["AUC"]})
comparison.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=45, title="Per-organ AUC: simulated vs real scans")
plt.axhline(0.5, color="grey", linestyle="--")
plt.ylabel("ROC AUC")
plt.tight_layout()
plt.show()

In [ ]:
# =================================
# Test-time augmentation without flips (base pass + 4 lightly zoomed/shifted passes)
# =================================
tta_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    zoom_range=0.08,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='constant',
    cval=0.0
)

def predict_tta(model, df, tta_steps=5):
    probs = predict_frame(model, df)
    for _ in range(tta_steps - 1):
        probs = probs + predict_frame(model, df, datagen=tta_datagen)
    return probs / tta_steps

aus_probs_tta = predict_tta(final_model, aus_test_df)
rus_probs_tta = predict_tta(final_model, rus_test_df)
score_set(aus_probs_tta, aus_test_df, EVAL_ORGANS["aus_test"], "AUS test (simulated), TTA")
score_set(rus_probs_tta, rus_test_df, EVAL_ORGANS["rus_test"], "RUS test (real), TTA")

In [ ]:
# =================================
# Qualitative FAST readout on unannotated real scans (no ground truth: not a score)
# =================================

def fast_window(prob_row):
    """Which abdominal FAST window the predicted organs suggest."""
    present = {o: prob_row[ORGANS.index(o)] >= thresholds[o] for o in ("liver", "kidney", "spleen")}
    if present["liver"] and present["kidney"]:
        return "RUQ window (liver + kidney, Morison's pouch)"
    if present["spleen"] and present["kidney"]:
        return "LUQ window (spleen + kidney, splenorenal recess)"
    return "no abdominal FAST window recognised"

gallery = rus_unlabeled_df.sample(n=min(8, len(rus_unlabeled_df)), random_state=SEED).reset_index(drop=True)
gallery_probs = predict_frame(final_model, gallery.assign(**{o: 0 for o in ORGANS}))

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, (_, row), probs in zip(axes.ravel(), gallery.iterrows(), gallery_probs):
    ax.imshow(cv2.imread(row["path"], cv2.IMREAD_GRAYSCALE), cmap="gray")
    top = sorted(zip(ORGANS, probs), key=lambda x: -x[1])[:3]
    ax.set_title(fast_window(probs) + "\n" + ", ".join(f"{o} {p:.2f}" for o, p in top), fontsize=9)
    ax.axis("off")
plt.suptitle("Qualitative only: real scans without annotations", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# =================================
# Save weights and thresholds (the ultrasound-trained backbone for stage 2)
# =================================
import json

save_path = "/kaggle/working/us_organ_recognition_weights"
final_model.save_weights(save_path)
with open("/kaggle/working/us_organ_recognition_config.json", "w") as fh:
    json.dump({"organs": ORGANS, "palette": {o: list(c) for o, c in PALETTE.items()},
               "thresholds": thresholds, "min_organ_frac": MIN_ORGAN_FRAC}, fh, indent=2)
print(f"Weights saved to {save_path} (TF checkpoint format; rebuild with build_vgg16_vit_model(len(ORGANS)) + load_weights)")